# Module 2: Clinical Data Analysis

This notebook explores the METABRIC clinical dataset for the Breast Cancer project. It reports data structure, missingness, clinical category distributions, numerical feature summaries and possible data-quality concerns.

**No machine-learning model is trained.** The source dataset is read only and is not modified.

## 1. Import libraries and load the tab-separated dataset

The file extension ends in `.txt`, but its contents are tab-separated. Keep this notebook and `Clinical_dataset.xlsx.txt` in the same folder, then run the notebook with that folder as the working directory.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Use a clean, readable plotting style.
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

# The source file is in the same Module 2 folder as this notebook.
DATA_PATH = Path("Clinical_dataset.xlsx.txt")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. "
        "Open this notebook with the Module 2 folder as the working directory."
    )

# Read tabs as the column separator. Treat common text markers as missing values.
df = pd.read_csv(
    DATA_PATH,
    sep="\t",
    na_values=["NA", "N/A", ""],
    low_memory=False
)

print(f"Loaded dataset from: {DATA_PATH.resolve()}")
print("The source file has only been read; it has not been changed.")

## 2. Dataset shape, columns, data types and missing values

The missing-value table shows the number and percentage of missing entries in every column.

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

print("\nColumn names:")
for column in df.columns:
    print(f"- {column}")

print("\nData types:")
display(df.dtypes.rename("Data type").to_frame())

missing_summary = pd.DataFrame({
    "Missing count": df.isna().sum(),
    "Missing percent": (df.isna().mean() * 100).round(2)
}).sort_values("Missing percent", ascending=False)

print("\nMissing values by column:")
display(missing_summary)

## 3. Neoplasm Histologic Grade distribution

Grades 1, 2 and 3 are shown as separate categories. Missing grade entries are displayed too, so the chart accounts for every row.

In [ ]:
grade_column = "Neoplasm Histologic Grade"

# Convert to labels for readable plotting, while preserving missing entries.
grade_labels = df[grade_column].map(
    lambda value: str(int(value)) if pd.notna(value) else "Missing"
)
grade_order = ["1", "2", "3", "Missing"]
grade_counts = grade_labels.value_counts().reindex(grade_order, fill_value=0)

print("Grade counts:")
display(grade_counts.rename("Rows").to_frame())

ax = grade_counts.plot(kind="bar", color="#4C78A8")
ax.set_title("Neoplasm Histologic Grade")
ax.set_xlabel("Histologic grade")
ax.set_ylabel("Number of rows")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Tumor Stage distribution

This chart includes missing stage values as their own category. Stage is displayed as recorded in the dataset.

In [ ]:
stage_column = "Tumor Stage"

stage_labels = df[stage_column].map(
    lambda value: str(int(value)) if pd.notna(value) and float(value).is_integer()
    else (str(value) if pd.notna(value) else "Missing")
)
stage_order = sorted(
    [value for value in stage_labels.unique() if value != "Missing"],
    key=lambda value: float(value)
)
if "Missing" in stage_labels.values:
    stage_order.append("Missing")

stage_counts = stage_labels.value_counts().reindex(stage_order, fill_value=0)

print("Tumor Stage counts:")
display(stage_counts.rename("Rows").to_frame())

ax = stage_counts.plot(kind="bar", color="#F58518")
ax.set_title("Tumor Stage Distribution (including missing values)")
ax.set_xlabel("Tumor stage")
ax.set_ylabel("Number of rows")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. ER, PR and HER2 status

Each bar chart includes missing values as an explicit category.

In [ ]:
status_columns = ["ER Status", "PR Status", "HER2 Status"]

fig, axes = plt.subplots(1, len(status_columns), figsize=(16, 5))
for ax, column in zip(axes, status_columns):
    counts = (
        df[column]
        .fillna("Missing")
        .astype(str)
        .value_counts()
        .sort_index()
    )
    counts.plot(kind="bar", ax=ax, color="#54A24B")
    ax.set_title(column)
    ax.set_xlabel("Recorded status")
    ax.set_ylabel("Number of rows")
    ax.tick_params(axis="x", rotation=35)

fig.suptitle("Hormone Receptor and HER2 Status", y=1.03)
plt.tight_layout()
plt.show()

## 6. Numerical feature summaries

The requested numerical features are summarized with count, mean, standard deviation, quartiles and range. Histograms show each feature's distribution; boxplots help spot skew and possible outliers. These plots do not establish whether a value is clinically incorrect.

In [ ]:
numeric_features = [
    "Age at Diagnosis",
    "Tumor Size",
    "Lymph nodes examined positive",
    "Mutation Count",
    "Nottingham prognostic index",
]

# Convert selected values to numeric where possible; unexpected text becomes missing.
for column in numeric_features:
    df[column] = pd.to_numeric(df[column], errors="coerce")

print("Numerical feature summary:")
display(df[numeric_features].describe().T)

# Plot one histogram per numerical feature.
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, column in zip(axes.flat, numeric_features):
    sns.histplot(df[column].dropna(), bins=30, kde=True, ax=ax, color="#4C78A8")
    ax.set_title(column)
    ax.set_xlabel(column)
    ax.set_ylabel("Number of rows")

# Hide the unused sixth subplot.
axes.flat[-1].axis("off")
fig.suptitle("Numerical Feature Histograms", y=1.02)
plt.tight_layout()
plt.show()

# Plot one boxplot per feature to compare medians, spread and potential outliers.
fig, axes = plt.subplots(1, len(numeric_features), figsize=(18, 5))
for ax, column in zip(axes, numeric_features):
    sns.boxplot(y=df[column], ax=ax, color="#72B7B2")
    ax.set_title(column)
    ax.set_ylabel("Value")

fig.suptitle("Numerical Feature Boxplots", y=1.03)
plt.tight_layout()
plt.show()

## 7. Correlation heatmap

The heatmap uses pairwise correlations among numerical columns. Correlation describes linear association in this dataset; it does not show that one feature causes another.

In [ ]:
numeric_data = df.select_dtypes(include="number")
correlation = numeric_data.corr()

plt.figure(figsize=(14, 11))
sns.heatmap(
    correlation,
    cmap="coolwarm",
    center=0,
    annot=True,
    fmt=".2f",
    square=True,
    linewidths=0.4
)
plt.title("Correlation Heatmap of Numerical Columns")
plt.tight_layout()
plt.show()

## 8. Data-quality checks and highly missing columns

The checks below flag missingness above 20%, duplicate records or identifiers, unexpected grade/stage values, and suspicious numeric ranges. Flags are prompts for review, not automatic corrections.

In [ ]:
# List columns with more than 20% missing values.
high_missing = missing_summary[missing_summary["Missing percent"] > 20]

print("Columns with more than 20% missing values:")
if high_missing.empty:
    print("None")
else:
    display(high_missing)

# Check for duplicate rows and repeated patient/sample identifiers.
print(f"\nCompletely duplicated rows: {int(df.duplicated().sum())}")
for id_column in ["Patient ID", "Sample ID"]:
    if id_column in df.columns:
        duplicate_rows = int(df[id_column].duplicated(keep=False).sum())
        repeated_ids = int((df[id_column].value_counts(dropna=True) > 1).sum())
        print(
            f"{id_column}: {df[id_column].nunique(dropna=True):,} unique non-missing IDs; "
            f"{repeated_ids:,} IDs occur more than once; "
            f"{duplicate_rows:,} rows have a repeated ID."
        )

# Check whether grade and stage values are within expected recorded categories.
observed_grades = sorted(df[grade_column].dropna().unique().tolist())
unexpected_grades = [value for value in observed_grades if value not in [1, 2, 3]]
observed_stages = sorted(df[stage_column].dropna().unique().tolist())
unexpected_stages = [value for value in observed_stages if value not in [0, 1, 2, 3, 4]]

print(f"\nObserved non-missing grades: {observed_grades}")
print(f"Unexpected grades outside 1, 2, 3: {unexpected_grades}")
print(f"Observed non-missing stages: {observed_stages}")
print(f"Unexpected stages outside 0–4: {unexpected_stages}")

# Simple range checks to help identify values that may need source review.
range_checks = {
    "Age at Diagnosis": (0, 120),
    "Tumor Size": (0, None),
    "Lymph nodes examined positive": (0, None),
    "Mutation Count": (0, None),
    "Nottingham prognostic index": (0, None),
}

print("\nValues outside simple review ranges:")
for column, (minimum, maximum) in range_checks.items():
    values = df[column]
    outside = pd.Series(False, index=df.index)
    if minimum is not None:
        outside |= values.notna() & (values < minimum)
    if maximum is not None:
        outside |= values.notna() & (values > maximum)
    print(f"{column}: {int(outside.sum())} row(s)")

## 9. Notes for interpretation

- Missing-value percentages depend on the missing markers recognized when the file was loaded (`NA`, `N/A`, and empty cells).
- Clinical columns may have different amounts of missingness because not every field is recorded for every patient or cohort.
- Check suspicious values against the dataset documentation before changing or excluding records.
- This notebook performs exploratory data analysis only. It does not train or evaluate a predictive model.